# Using Apple Store Reviews in Machines of Knowledge

One type of data with which you may want to work as part of the *Machines of Knowledge* course in the Master Digital Cultures, or in your MA thesis, are reviews scraped from the Apple Store.

The following code will allow you to collect the review text alongside important metadata like the star ratings.

**Note:** This notebook is designed to run via Live Code, directly in your browser. There is no shared server, so each person defines their own podcasts, runs the scraper themselves, and downloads their own results at the end — nothing is saved automatically.

## Apple Podcast Review Scraping with the app_store_scraper

This program is a wrapper for scraping Apple Podcast Reviews with **app-store-web-scraper** (thank you Eric Lim, see https://pypi.org/project/app-store-scraper/, MIT license). It was adapted for use in teaching at Maastricht University by Monika Barget and Arnoud Wils in 2023, and further adapted for in-browser Live Code execution.

The main script is kept as lean as possible to make it easy to use for students without previous coding experience. To use the script, carefully read and follow the instructions below.

## Install and import modules

This section ensures that your script has all the necessary functionalities. Just select the grey box below and click on the black arrow in the tool bar. Wait for the completion message before you continue!

**Note:** Live Code runs Python inside your browser via Pyodide, so packages are installed with `micropip`, and `requests`/`urllib3` need to be patched (via `pyodide-http`) so they can make network requests through the browser.

In [ ]:
await micropip.install("pyodide-http")
await micropip.install("app-store-web-scraper")

import pyodide_http
pyodide_http.patch_all()  # let requests/urllib3 work through the browser

from pprint import pprint
import os
import glob
import re
import pandas as pd
import urllib3
import numpy as np

print("Installations and package import complete!")

## Load the helper modules

This script relies on three small helper files (`verify_countries.py`, `scrape_reviews.py`, and `applestore_country_codes.py`) that live in the `book/python` folder of the TeachBooks repository. Since Live Code has no access to the repository's filesystem, the cell below downloads the raw text of each file over the network and writes it into the browser's in-memory filesystem, from where it can be imported normally.

In [ ]:
MODULES_BASE_URL = "https://raw.githubusercontent.com/MaastrichtUniversityPress/distant-reading-textbook/main/book/python"

helper_files = ["verify_countries.py", "scrape_reviews.py", "applestore_country_codes.py"]

from pyodide.http import pyfetch

async def fetch_module(filename):
    response = await pyfetch(f"{MODULES_BASE_URL}/{filename}")
    text = await response.string()
    with open(filename, "w") as f:
        f.write(text)

for fname in helper_files:
    await fetch_module(fname)

from verify_countries import pool_checks
from scrape_reviews import scrape_reviews
from applestore_country_codes import select_countries

print("Helper modules loaded!")

## Define data input

In this section, you may need to adjust a few things, depending on your research project. The box below loads Apple Store country codes. It is recommended to use all available country codes, but if you want to limit them, adjust the selection inside `applestore_country_codes.py`.

In [ ]:
countries = select_countries()

print("The country codes have been successfully loaded:", countries)

Next, define the podcasts you want to scrape reviews for. Replace the app IDs and app names below with your own selected podcasts — the existing values are just a test to check that the script works.

In [ ]:
# Define a list of App Store items with app_id and app_name for scraping
# remove or add lines within the podcast list if needed
podcasts = [
    {"app_id": 1568547321, "app_name": "are-you-menstrual"},
    {"app_id": 1614435903, "app_name": "28ish-days-later"},
    {"app_id": 1537830674, "app_name": "holistic-womens-health-hormones-endometriosis-pcos"},
]

# URL structure of a typical App Store item:
# https://podcasts.apple.com/us/podcast/black-women-talk-tech-podcast/id1453181438
# copy the ID and podcast name from your own URL

# Standard URL for Apple Podcasts
base_url = "https://podcasts.apple.com/us/podcast/"

# Important: country codes are selected from the list loaded above

# Output folder (in the browser's in-memory filesystem)
path_out = "output/"
os.makedirs(path_out, exist_ok=True)

print("Podcasts defined!")

## Validate data and collect reviews

Here, you only need to run the code below and monitor the output. No changes in the script are required from your side.

**A note on running this in the browser:** this step makes real network requests to Apple's servers. Some browsers and network setups block these cross-origin requests (a security feature called CORS) even after patching `requests` for Pyodide. If you see repeated connection errors here rather than the usual per-country scraping messages, let your tutor know.

In [ ]:
# Loop through podcasts and country codes

for podcast in podcasts:
    app_id = podcast["app_id"]
    app_name = podcast["app_name"]

    # Construct full URL for each podcast
    podcast_url = f"{base_url}{app_name}/id{app_id}"
    print(f"Scraping URL: {podcast_url}")

    # Create the filename and path for each podcast's review file
    filename_csv = f"{app_name}_reviews_table.csv"
    file_csv = os.path.join(path_out, filename_csv)
    print(f"Saving reviews to: {file_csv}")

    # Check available countries and get the list of country codes
    countries_reviewed = pool_checks(podcast_url, countries)
    print("The following countries have reviews:", countries_reviewed)

    # Collect all reviews for selected countries using scrape_reviews
    all_reviews = scrape_reviews(countries_reviewed, app_name, app_id)
    print("All reviews collected for ", podcast, "!")

    # Only proceed to save if reviews were actually collected
    if all_reviews:
        try:
            # Concatenate all country-specific DataFrames into one DataFrame per podcast
            combined_reviews_df = pd.concat(all_reviews, ignore_index=True)

            # Save the combined DataFrame for each podcast
            combined_reviews_df.to_csv(file_csv, index=False)
            print(f"Reviews saved successfully to {file_csv}")

        except Exception as e:
            print(f"An error occurred while saving reviews for {app_name}: {e}")
    else:
        print(f"No reviews collected for {app_name}. Skipping save.")

# After creating all individual CSV files, merge them

output_filename = "all_reviews_table"

# Check if output file exists from previous script execution
existing_files = glob.glob(os.path.join(path_out, f"{output_filename}*.csv"))
if existing_files:
    max_index = max(
        [
            int(os.path.splitext(os.path.basename(f))[0].split("_")[-1])
            for f in existing_files if f"{output_filename}_" in f
        ] + [1]
    )
    new_filename = f"{output_filename}_{max_index + 1}.csv"
else:
    new_filename = f"{output_filename}.csv"  # First file

file_csv2 = os.path.join(path_out, new_filename)

# Exclude existing all_reviews_table.csv
all_files = glob.glob(os.path.join(path_out, "*_reviews_table.csv"))
all_files = [f for f in all_files if os.path.basename(f) != "all_reviews_table.csv"]

if all_files:
    # Combine all review files
    combined_df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)

    # Save the combined DataFrame to the new CSV file
    print("Your final dataframe has", len(combined_df), "rows.")
    combined_df.to_csv(file_csv2, index=False)
    print(f"Exported to {file_csv2}")
else:
    print("No review files found to combine.")

# NOTE: the review count seen on the landing page of a podcast differs from the actual number of reviews fetched.
# This is simply because only some users who rated the app also leave reviews.

## Download your results

Because Live Code runs inside your browser, there's no shared `/output` folder you can browse to afterwards — the cell below builds a download link for every CSV file that was just created, so you can save your results to your own computer.

In [ ]:
from IPython.display import display, HTML
import base64

csv_files = sorted(glob.glob(os.path.join(path_out, "*.csv")))

if not csv_files:
    print("No CSV files found yet — run the scraping step above first.")
else:
    for csv_file in csv_files:
        with open(csv_file, "rb") as f:
            data = f.read()
        b64 = base64.b64encode(data).decode("utf-8")
        filename = os.path.basename(csv_file)
        href = f'<a download="{filename}" href="data:text/csv;base64,{b64}" target="_blank">\u2b07\ufe0f Download {filename}</a>'
        display(HTML(href))

Make sure to restart the kernel (circle button above) before entering new podcast URLs and running the script again. Otherwise you may see old data copied to your new files.

## Further Readings

- Fuchs, C., Hofkirchner, W., Schafranek, M., Raffl, C., Sandoval, M., & Bichler, R. (2010). Theoretical foundations of the web: Cognition, communication, and co-operation. Towards an understanding of Web 1.0, 2.0, 3.0. *Future Internet*, 2(1), 41–59. https://doi.org/10.3390/fi2010041
- Levmore, S., & Nussbaum, M. C. (Eds.). (2010). *The offensive internet: Speech, privacy, and reputation.* Harvard University Press.
- McGregor, H. (2022). Podcast studies. In *Oxford Research Encyclopedia of Literature*. Oxford University Press. https://doi.org/10.1093/acrefore/9780190201098.013.1338
- Wendland, J. (2024). Building a better participatory culture and enhancing sense of community in podcasts: Systematic literature review. *Journal of Radio & Audio Media*, 1–23. https://doi.org/10.1080/19376529.2024.2347609
- Wendland, J. (2025). Social podcasting – Levels of podcast audience participation. *Journal of Radio & Audio Media*, 1–25. https://doi.org/10.1080/19376529.2025.2460834